# 01 - Getting Started with the ATLAS Automation API

Connect to a running ATLAS instance from Python, list the loaded sessions, read a parameter, and plot it.

**Before you run:** start ATLAS 11+ and load a session containing `vCar:Chassis` into Set 1.

## Load the Automation API

We start the CoreCLR runtime that ATLAS ships with, then reference the two Automation API assemblies.

In [ ]:
import os
from pythonnet import load

ATLAS_DIR = os.environ.get(
    "ATLAS_DIR", r"C:\Program Files\McLaren Applied Technologies\ATLAS 10"
)
load("coreclr", runtime_config=os.path.join(ATLAS_DIR, "MAT.Atlas.Host.runtimeconfig.json"))

import clr
clr.AddReference(os.path.join(ATLAS_DIR, "MAT.Atlas.Automation.Api.dll"))
clr.AddReference(os.path.join(ATLAS_DIR, "MAT.Atlas.Automation.Client.dll"))

from MAT.Atlas.Automation.Api.Enums import DataStatusType
from MAT.Atlas.Automation.Client.Services import (
    ApplicationServiceClient,
    ParameterDataAccessServiceClient,
    SessionServiceClient,
    SetServiceClient,
    WorkbookServiceClient,
)
print("Automation API loaded.")

## Connect to ATLAS

In [ ]:
application = ApplicationServiceClient("Notebook")
application.Connect("Notebook")
print(f"Connected to ATLAS {application.GetVersion()}")

## List sets and sessions

In [ ]:
workbook = WorkbookServiceClient()
sets = SetServiceClient()

set_list = workbook.GetSets()
assert len(set_list) > 0, "No sets configured - load a session into Set 1."

for s in set_list:
    print(f"Set '{s.Name}' (parameters: {s.Parameters})")
    for session in sets.GetCompositeSessions(s.Id):
        print(f"    {session.Name}  [{session.State}]")

session_id = sets.GetCompositeSessions(set_list[0].Id)[0].Id

## Read a parameter

In [ ]:
PARAMETER = "vCar:Chassis"

session_service = SessionServiceClient()
pda = ParameterDataAccessServiceClient()

parameter = session_service.GetSessionParameter(session_id, PARAMETER)
timebase = session_service.GetSessionTimeBase(session_id)
estimate = pda.GetSamplesCountEstimate(session_id, parameter.Identifier, timebase.StartTime, timebase.EndTime)

pda.Goto(session_id, parameter.Identifier, timebase.StartTime)
samples = pda.GetNextSamples(session_id, parameter.Identifier, estimate)

# Convert to seconds (relative to the start) and keep only real samples.
start = timebase.StartTime
times, values = [], []
for i in range(samples.SampleCount):
    if samples.DataStatus[i].HasFlag(DataStatusType.Sample):
        times.append((samples.Time[i] - start) / 1e9)
        values.append(samples.Data[i])

print(f"Read {len(values)} samples of {parameter.Identifier} ({parameter.Units}).")

## Plot it

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(times, values, linewidth=0.8)
plt.xlabel("Time (s)")
plt.ylabel(f"{parameter.Name} ({parameter.Units})")
plt.title(parameter.Identifier)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Tidy up the clients.
for client in (application, workbook, sets, session_service, pda):
    client.Dispose()